In [13]:
import fitz  # PyMuPDF
from PIL import Image
import numpy as np
import os

def process_pdf_extract_images_and_save_high_res(pdf_path, padding=20, tolerance=245, dpi=300):

    output_folder="haha"
    output_pdf_path = "haha.pdf"
    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    doc = fitz.open(pdf_path)
    images = []
    
    # Process the first page to get the bounding box with high DPI
    first_page = doc.load_page(0)
    first_pix = first_page.get_pixmap(dpi=dpi)
    first_img = Image.frombytes("RGB", [first_pix.width, first_pix.height], first_pix.samples)
    
    # Convert the first image to numpy array
    img_array = np.array(first_img)
    
    # If it's an RGB image, convert to grayscale for easier comparison
    if img_array.ndim == 3:
        img_gray = np.mean(img_array, axis=2)
    else:
        img_gray = img_array
    
    # Create a mask to identify non-white pixels based on tolerance
    mask = img_gray < tolerance
    
    # Find the coordinates of the non-white pixels
    coords = np.argwhere(mask)
    
    # If no non-white pixels found, raise an exception
    if coords.size == 0:
        raise ValueError("No non-white content found on the first page.")
    
    # Get the bounding box of the non-white pixels
    (x0, y0), (x1, y1) = coords.min(axis=0), coords.max(axis=0) + 1
    
    # Iterate through all the pages and apply the bounding box with padding, keeping high resolution
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        # Apply the bounding box with padding
        img_array = np.array(img)
        x0_padded = max(x0 - padding, 0)
        y0_padded = max(y0 - padding, 0)
        x1_padded = min(x1 + padding, img_array.shape[0])
        y1_padded = min(y1 + padding, img_array.shape[1])
        
        cropped_img = img.crop((y0_padded, x0_padded, y1_padded, x1_padded))
        
        # Save the image locally in high resolution
        img_path = os.path.join(output_folder, f"page_{page_num + 1}.png")
        cropped_img.save(img_path, dpi=(dpi, dpi))
        
        images.append(cropped_img)
    
    # Save the processed images as a high-resolution PDF
    if images:
        images[0].save(output_pdf_path, save_all=True, append_images=images[1:], resolution=dpi)
    
    return output_pdf_path

pdf_path = "reish main 1-24-25.pdf"

# Process the PDF, save images, and get the output PDF path in high resolution
processed_pdf = process_pdf_extract_images_and_save_high_res(pdf_path)
print(f"Processed high-res PDF saved at: {processed_pdf}")

Processed high-res PDF saved at: haha.pdf


In [5]:
import os
import fitz
from PIL import Image
import numpy as np

In [8]:
import fitz  # PyMuPDF
from PIL import Image
i
import numpy as np

def get_bounding_box(image, tolerance=245):
    # Convert the image to numpy array
    img = np.array(image)
    
    # If it's an RGB image, convert to grayscale for easier comparison
    if img.ndim == 3:
        img_gray = np.mean(img, axis=2)
    else:
        img_gray = img
    
    # Create a mask to identify non-white pixels based on tolerance
    mask = img_gray < tolerance
    
    # Find the coordinates of the non-white pixels
    coords = np.argwhere(mask)
    
    # If no non-white pixels found, return the original image's bounds
    if coords.size == 0:
        return None
    
    # Get the bounding box of the non-white pixels
    x0, y0 = coords.min(axis=0)
    x1, y1 = coords.max(axis=0) + 1
    
    return x0, y0, x1, y1

# def remove_white_margins_with_bounding_box(image, bounding_box, padding=10):
#     # Extract bounding box with padding
#     x0, y0, x1, y1 = bounding_box
    
#     # Convert image to numpy array to get shape for bounds checking
#     img = np.array(image)
    
#     # Add padding and ensure the values are within the image bounds
#     x0 = max(x0 - padding, 0)
#     y0 = max(y0 - padding, 0)
#     x1 = min(x1 + padding, img.shape[0])
#     y1 = min(y1 + padding, img.shape[1])
    
#     # Crop the image based on the bounding box with padding
#     return image.crop((y0, x0, y1, x1))

def extract_images_from_pdf_using_first_page_margin(pdf_path, padding=10):
    doc = fitz.open(pdf_path)
    images = []
    
    # Process the first page to get the bounding box
    first_page = doc.load_page(0)
    first_pix = first_page.get_pixmap()
    first_img = Image.frombytes("RGB", [first_pix.width, first_pix.height], first_pix.samples)
    bounding_box = get_bounding_box(first_img)
    
    if bounding_box is None:
        raise ValueError("No non-white content found on the first page.")
    
    # Apply the same bounding box to all pages
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img_no_margins = remove_white_margins_with_bounding_box(img, bounding_box, padding=padding)
        images.append(img_no_margins)
    
    return images

def save_images_to_pdf(images, output_pdf_path):
    if images:
        images[0].save(output_pdf_path, save_all=True, append_images=images[1:])

# Correct paths for your PDFs
pdf1_path = "test27.pdf"
output_pdf1 = "processed_test27.pdf"

pdf2_path = "reish main 1-24-25.pdf"
output_pdf2 = "processed_reish_main_1-24-25.pdf"

# Process first PDF using first page's margin info for all pages
pdf1_images = extract_images_from_pdf_using_first_page_margin(pdf1_path, padding=10)
save_images_to_pdf(pdf1_images, output_pdf1)

# Process second PDF using first page's margin info for all pages
pdf2_images = extract_images_from_pdf_using_first_page_margin(pdf2_path, padding=10)
save_images_to_pdf(pdf2_images, output_pdf2)

print("Processing complete. PDFs saved as processed_test27.pdf and processed_reish_main_1-24-25.pdf.")


Processing complete. PDFs saved as processed_test27.pdf and processed_reish_main_1-24-25.pdf.
